In [ ]:

# package
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

from helper import *




In [ ]:
# LLM-based MITI 4.2.1 scores for individual interviews
scores = pd.read_csv("/path/to/project/data/processed/main_social_media/w2_miti_global_scores_20251222_v003.csv")
# Cleaned survey data for getting treatment assignment
survey = pd.read_stata("/path/to/project/data/processed/main_social_media/clean_merged.dta")

# Merge data
scores["user_id_raw"] = scores["session_id"].str[11:].astype(int)
data = survey[["user_id_raw", "T"]].merge(scores, on=["user_id_raw"], how="left")
data = data.dropna(subset=["score", "miti_dimension", "T", "user_id_raw"])
data["T"] = data["T"].replace("Ambivalence", "Decisional Balance")

In [ ]:
data.head()

In [ ]:
df_plot = data.copy()
df_plot["T"] = df_plot["T"].astype(str).str.strip()

df_plot = df_plot[df_plot["T"].isin(["Decisional Balance", "Change Talk"])]

df_plot["score"] = np.where(
    (df_plot["score"] < 0) & (pd.isna(df_plot["gpt_response"])),
    pd.NA,
    df_plot["score"]
)

In [ ]:
df_plot["score"].describe() 

In [ ]:
df_error = df_plot[df_plot["score"] < 0]

In [ ]:
df_plot.head()

means_t1 = df_plot[df_plot["T"] == "Decisional Balance"].groupby(["miti_dimension"])["score"].mean()
means_t2 = df_plot[df_plot["T"] == "Change Talk"].groupby(["miti_dimension"])["score"].mean()

print("Means Decisional Balance:")
print(means_t1)
print("\nMeans Change Talk:")
print(means_t2)

In [ ]:


def plot_percent_bars_by_miti_dimension(
    df, value_col, t_col="T", dim_col="miti_dimension",
    show_means=True
):

    d = df.copy()
    dims = list(pd.unique(d[dim_col]))[:4]

    set_plot_theme()

    fig, axes = plt.subplots(
        2, 2, figsize=(12, 8),
        sharex=True, sharey="row",
        constrained_layout=True
    )
    axes = axes.ravel()

    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    # bins centered on integer scores 0..5
    bins = np.arange(-0.5, 5.6, 1)

    for i, ax in enumerate(axes):
        if i >= len(dims):
            ax.axis("off")
            continue

        dim = dims[i]
        sub = d[d[dim_col] == dim]

        sns.histplot(
            data=sub,
            x=value_col,
            hue=t_col,
            bins=bins,
            multiple="dodge",     # side-by-side bars
            shrink=0.85,
            stat="percent",       # bar heights sum to 100
            common_norm=False,    # normalize each hue level independently
            discrete=True,
            element="bars",
            edgecolor="white",
            linewidth=0.5,
            alpha=0.85,
            ax=ax,
            legend=(i == 0)
        )

        # --- X axis: ticks 1..5 only; show tick labels on all subplots ---
        ax.set_xlim(0.5, 5.5)
        ax.set_xticks([1, 2, 3, 4, 5])

        # Show tick labels even on the top row when sharex=True
        ax.tick_params(axis="x", labelbottom=True)

        # X-axis title only on bottom row (no title on top row)
        if i in (2, 3):  # bottom row in a 2x2 layout
            ax.set_xlabel("MITI-Score", fontsize=13)
        else:
            ax.set_xlabel("")

        ax.set_ylabel("Percent", fontsize=13)

        if show_means:
            t_values = list(pd.unique(sub[t_col]))
            y_position = 0.5
            for j, t_val in enumerate(t_values):
                vals = sub.loc[sub[t_col] == t_val, value_col].dropna()
                if len(vals) == 0:
                    continue
                m = vals.mean()
                ax.text(
                    0.5, y_position, f"{t_val} Mean: {m:.2f}",
                    transform=ax.transAxes,
                    color=color_cycle[j % len(color_cycle)],
                    fontsize=12,
                    fontweight="bold",
                    ha="right", va="top"
                )
                y_position -= 0.08

        # Legend only on top-left
        if i == 0:
            leg = ax.get_legend()
            if leg is not None:
                leg.set_title("")          # remove title
                leg.set_frame_on(True)
                leg.get_frame().set_alpha(1)
                leg.get_frame().set_linewidth(1)
        else:
            leg = ax.get_legend()
            if leg is not None:
                leg.remove()

        ax.set_title(f"Dimension: {dim}", fontsize=14, fontweight="bold")
        ax.tick_params(labelsize=11)
        finalize_plot(ax)

    return fig

In [ ]:
fig = plot_percent_bars_by_miti_dimension(
    df_plot,
    value_col="score"
)

fig.savefig(
    "/path/to/project/code/analysis_NR/6731ca401220dcd3b28dc2ec/figures/fig_miti_score_histograms.pdf")
